# GENE linear scan summary

The notebook form of `scripts/scan_summary.py`. For a scan over ky and any number
of other parameters it

1. parses `scan.log`,
2. plots growth rate and frequency against ky, **one figure per parameter combination**,
3. reads each run's `nrg` file for `Q_es(electrons)/Q_es(ions)` and `Q_em/Q_es` for electrons.

The parsing is imported from the script rather than copied, so the two cannot drift
apart — `scan.log` comes in more than one shape and the header is easy to get wrong.
The plotting is written out inline here, because that is the part worth tweaking.

Set `SCAN_DIR` and run the cells.

In [ ]:
%matplotlib inline
import importlib.util, sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SCAN_DIR = "/path/to/scanfiles0001"   # <-- edit me
NAVG     = 1      # nrg steps to average the flux ratios over; 1 = the last step

# Locate scripts/scan_summary.py. Notebooks do not reliably know their own path,
# so try the usual spots and say clearly when none of them hold it.
def _load_scan_summary():
    here = Path.cwd()
    candidates = [p / "scripts" / "scan_summary.py"
                  for p in (here, here.parent, here.parent.parent)]
    for path in candidates:
        if path.is_file():
            spec = importlib.util.spec_from_file_location("scan_summary", path)
            mod = importlib.util.module_from_spec(spec)
            spec.loader.exec_module(mod)
            return mod, path
    raise FileNotFoundError(
        "scripts/scan_summary.py not found; looked in:\n  "
        + "\n  ".join(str(p) for p in candidates)
        + "\nSet the path by hand if the repo lives elsewhere.")

scan_summary, _script = _load_scan_summary()
print("using", _script)

## 1. Parse `scan.log`

`parse_scan_log` returns one entry per run — the run number, every scanned
parameter, and the eigenvalue — plus the list of parameter names. The ky column is
found by name (`kymin`, `ky`, ...); pass it explicitly to `identify_ky` if your
scan calls it something else.

In [ ]:
scan_dir = Path(SCAN_DIR)
log_path = scan_dir / "scan.log" if scan_dir.is_dir() else scan_dir
scan_dir = log_path.parent

entries, param_names = scan_summary.parse_scan_log(log_path)
ky_col = scan_summary.identify_ky(param_names)
others = [p for p in param_names if p != ky_col]

print(f"{len(entries)} runs")
print(f"scanned parameters: {param_names}")
print(f"ky column: {ky_col!r}; grouping by: {others or '(nothing — ky-only scan)'}")

## 2. Fetch the flux ratios from each run's `nrg`

The ratio is formed **per time step and then averaged** over the last `NAVG`
steps, not as a ratio of averages: in a linear run both fluxes grow
exponentially, so a mean of either one alone is just its final step.

Electrons are identified by `charge < 0` in the run's parameters file. A run whose
`nrg` is missing gets `NaN` rather than a guess.

In [ ]:
rows = []
missing = []
for e in entries:
    row = {"run": e["run"], ky_col: e["params"].get(ky_col, np.nan),
           "gamma": e["gamma"], "omega": e["omega"]}
    row.update({p: e["params"].get(p) for p in others})
    row["Qes_e/Qes_i"] = np.nan
    row["Qem/Qes_e"] = np.nan

    nrg = scan_summary.find_run_file(str(scan_dir), "nrg", e["run"])
    if nrg is None:
        missing.append(e["run"])
    else:
        species = scan_summary.species_order(str(scan_dir), e["run"])
        r = scan_summary.flux_ratios(nrg, species, NAVG)
        row["Qes_e/Qes_i"] = r["q_es_e_over_i"]
        row["Qem/Qes_e"] = r["q_em_over_es_e"]

    rows.append(row)

df = pd.DataFrame(rows).sort_values(others + [ky_col]).reset_index(drop=True)
if missing:
    print(f"no nrg file for runs: {', '.join(missing)}")
df

## 3. Growth rate and frequency against ky

One figure per combination of the non-ky parameters — that is what "for every
parameter" means for a scan like `x0` x `kymin`.

In [ ]:
groups = df.groupby(others) if others else [((), df)]

for key, sub in groups:
    sub = sub.sort_values(ky_col)
    label = ", ".join(f"{p} = {v:g}" for p, v in zip(others, np.atleast_1d(key))) \
            if others else "linear scan"

    fig, ax = plt.subplots(1, 2, figsize=(11, 4))
    ax[0].plot(sub[ky_col], sub["gamma"], "o-")
    ax[0].set_ylabel(r"$\gamma\;[c_s/L_{ref}]$")
    ax[1].plot(sub[ky_col], sub["omega"], "s-", color="C1")
    ax[1].set_ylabel(r"$\omega\;[c_s/L_{ref}]$")
    ax[1].axhline(0.0, color="0.6", lw=0.8)
    for a in ax:
        a.set_xlabel(r"$k_y \rho_s$")
        a.grid(True, alpha=0.3)
    fig.suptitle(label)
    fig.tight_layout()
plt.show()

## 4. The heat-flux ratios against ky

`Q_es(e)/Q_es(i)` says how the heat flux splits between the species;
`Q_em(e)/Q_es(e)` says how electromagnetic the electron transport is. Reading them
beside `gamma(ky)` is the point — a mode that dominates the growth rate need not
dominate the transport, and the ratios often change character across the spectrum
where the branch changes.

In [ ]:
for key, sub in (df.groupby(others) if others else [((), df)]):
    sub = sub.sort_values(ky_col)
    label = ", ".join(f"{p} = {v:g}" for p, v in zip(others, np.atleast_1d(key))) \
            if others else "linear scan"

    fig, ax = plt.subplots(1, 2, figsize=(11, 4))
    ax[0].plot(sub[ky_col], sub["Qes_e/Qes_i"], "^-", color="C2")
    ax[0].set_ylabel(r"$Q_{es}^{e}/Q_{es}^{i}$")
    ax[1].plot(sub[ky_col], sub["Qem/Qes_e"], "v-", color="C3")
    ax[1].set_ylabel(r"$Q_{em}^{e}/Q_{es}^{e}$")
    ax[1].axhline(0.0, color="0.6", lw=0.8)
    for a in ax:
        a.set_xlabel(r"$k_y \rho_s$")
        a.grid(True, alpha=0.3)
    fig.suptitle(label)
    fig.tight_layout()
plt.show()

## 5. Everything on one axis

Growth rate against ky with one curve per parameter value, which is usually the
plot you actually want out of a two-parameter scan.

In [ ]:
if others:
    fig, ax = plt.subplots(1, 2, figsize=(11, 4))
    for key, sub in df.groupby(others):
        sub = sub.sort_values(ky_col)
        label = ", ".join(f"{p}={v:g}" for p, v in zip(others, np.atleast_1d(key)))
        ax[0].plot(sub[ky_col], sub["gamma"], "o-", ms=3, label=label)
        ax[1].plot(sub[ky_col], sub["omega"], "s-", ms=3, label=label)
    ax[0].set_ylabel(r"$\gamma\;[c_s/L_{ref}]$")
    ax[1].set_ylabel(r"$\omega\;[c_s/L_{ref}]$")
    ax[1].axhline(0.0, color="0.6", lw=0.8)
    for a in ax:
        a.set_xlabel(r"$k_y \rho_s$")
        a.grid(True, alpha=0.3)
        a.legend(fontsize=8)
    fig.tight_layout()
    plt.show()
else:
    print("ky-only scan: nothing to overlay.")

## 6. Look inside one run

The thing a script cannot give you: the `nrg` time traces behind a single point,
to check the run was actually converged before its ratio was taken. Set `RUN` to
any run number from the table above.

In [ ]:
RUN = entries[0]["run"]        # <-- edit me

nrg_path = scan_summary.find_run_file(str(scan_dir), "nrg", RUN)
times, data = scan_summary.read_nrg(nrg_path)
species = scan_summary.species_order(str(scan_dir), RUN)
names = [n for n, _ in species] or [f"species {i}" for i in range(data.shape[1])]

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
for i, name in enumerate(names):
    ax[0].semilogy(times, np.abs(data[:, i, scan_summary.NRG_Q_ES]), label=name)
    ax[1].plot(times, data[:, i, scan_summary.NRG_Q_EM]
               / np.where(data[:, i, scan_summary.NRG_Q_ES] != 0,
                          data[:, i, scan_summary.NRG_Q_ES], np.nan), label=name)
ax[0].set_ylabel(r"$|Q_{es}|$"); ax[0].set_title(f"run {RUN}")
ax[1].set_ylabel(r"$Q_{em}/Q_{es}$")
ax[1].set_title("ratio — flat means the eigenmode has settled")
for a in ax:
    a.set_xlabel(r"$t\;[L_{ref}/c_s]$")
    a.grid(True, alpha=0.3)
    a.legend(fontsize=8)
fig.tight_layout()
plt.show()

## 7. Save the table

In [ ]:
df.to_csv("scan_summary.csv", index=False)
print("wrote scan_summary.csv")